In [10]:
from src.components.optuna_model_selection.model import CorrectionModel

from torchvision import transforms
from src.entity.config_entity import ModelCreationEntity
from src.constants.local import CORRECTION_TEST_CSV
from pathlib import Path
from scipy.ndimage import gaussian_filter1d
import torch
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
import cv2

In [11]:
class DatasetForMultipleCorrectionLoops(Dataset):
    def __init__(self, subset_df_path, transform = None, random_state = 42, ):
        subset_df = pd.read_csv(subset_df_path).drop(['case', 'slice', 'prev_path', 'next_path'], axis = 1)

        
        self.transform = transform
        self.random_state = random_state
        summed_norm = np.array([74,61,45])
        self.summed_norm = np.expand_dims(summed_norm, (1,2))


        self.partial_df = subset_df
        

            
        

   

    
    def __len__(self):

        return len(self.partial_df)

            

    def __getitem__(self, index): 

        entry = self.partial_df.iloc[index]

        
        img = cv2.imread(entry.path, cv2.IMREAD_UNCHANGED)
        inp_mask = np.load(entry.model_mask_path, allow_pickle=True)['arr_0']
        out_mask = np.load(entry.out_transformed_mask_path, allow_pickle=True)['arr_0']
        summed_mask = np.load(entry.collapsed_mask_path, allow_pickle=True)['arr_0']
    
        

        min_img = img.min()
        max_img = img.max()
        img = (img - min_img)/(max_img - min_img + 1e-8)
        summed_mask = summed_mask/self.summed_norm

        img = torch.from_numpy(img).to(torch.float32)
        inp_mask = torch.from_numpy(inp_mask).to(torch.float32)
        out_mask = torch.from_numpy(out_mask).to(torch.float32)
        summed_mask = torch.from_numpy(summed_mask).to(torch.float32)

        if img.ndim == 2:
            img = img.unsqueeze(0)

        if self.transform:
            img = self.transform(img)

     

        return (img, inp_mask, summed_mask), entry.collapsed_id,  out_mask

    

    



In [12]:
class FinalModelCreationEntity(ModelCreationEntity):


    def __init__(self):
  
        super().__init__(safety_batch=False)

        
    
        self.inp_model_path=Path(f'data/models/correction_model.ckpt')
        #self.inp_model_path=Path('model-final (2).ckpt')
        #self.inp_model_path = path #Path(f'models/final_inp/model_{index}.ckpt')

        hparams = torch.load(self.inp_model_path, 'cpu', weights_only= False)['hyper_parameters']
        self.first_conv_out_channels = hparams['first_conv_out_channels']
        self.depth = hparams['depth']
        self.n_encoder_conv_layers = hparams['n_encoder_conv_layers']
        self.n_decoder_conv_layers = hparams['n_decoder_conv_layers']
        self.kernel_sizes = hparams['kernel_sizes']
        self.empty_mri_ratio = 0.1
        self.lr = hparams['learning_rate']
        self.w = hparams['weight_decay']
        self.inp_channels = 7



In [13]:


class F1Metric:
    def __init__(self, config, num_integ_bins=100001):
        self.device = torch.device('cuda' if config.accelerator == 'gpu' else 'cpu') 
        state_dict = torch.load(config.inp_model_path, 
                                map_location=self.device, weights_only=False)['state_dict']
        inner_state_dict = {
            k[len("model."):]: v for k, v in state_dict.items() if k.startswith("model.")
        }

        self.model = CorrectionModel(
            config.inp_channels,
            config.first_conv_out_channels,
            config.num_classes,
            config.depth,
            config.n_encoder_conv_layers,
            config.n_decoder_conv_layers,
            config.kernel_sizes
        )
        self.model.load_state_dict(inner_state_dict)
        dataset = DatasetForMultipleCorrectionLoops(
            CORRECTION_TEST_CSV,  
            transform=transforms.Compose([
                transforms.Resize(256),
                transforms.CenterCrop(256)
            ])
        )
            
        self.val_dataloader = DataLoader(
            dataset, batch_size=config.batch_size
        ) 
        
        self.num_channels = 3
        self.num_integ_bins = num_integ_bins

        summed_norm = np.array([74, 61, 45])
        # FIXED: Added the missing closing parenthesis
        self.summed_norm = np.expand_dims(summed_norm, (1, 2))
        
        # Convert to tensor once for fast GPU division later
        self.norm_tensor = torch.tensor(self.summed_norm, dtype=torch.float32, device=self.device)


    def calculate_f1_direct(self, dataloader, order, thresholds):
        self.model.to(self.device)
        self.model.eval()
       
        thresh_tensor = torch.tensor(thresholds, device=self.device).view(1, self.num_channels, 1, 1)
        
        total_tp = torch.zeros(self.num_channels, device=self.device)
        total_fp = torch.zeros(self.num_channels, device=self.device)
        total_fn = torch.zeros(self.num_channels, device=self.device)
        
        with torch.no_grad():
            if order == 0:

                for i, batch in enumerate(dataloader):
                    inp, index, mask = batch
                    mask = mask.to(self.device) 
                    
                    # Ensure inp[1] is on device
                    out = torch.sigmoid(inp[1].to(self.device))
                    preds = (out >= thresh_tensor).float()
                                        
                    total_tp += (preds * mask).sum(dim=(0, 2, 3))
                    total_fp += (preds * (1 - mask)).sum(dim=(0, 2, 3))
                    total_fn += ((1 - preds) * mask).sum(dim=(0, 2, 3))
                    
                    if i % 10 == 0:
                        print(f"{i} batches processed")
                        
            else:
                model_masks = []
                indexes = []
                masks = []
                imgs = []
                prev_summed_masks = None
                
                for step in range(order):
                    
                    summed_masks = torch.zeros((300, self.num_channels, 256, 256), device=self.device)
                    current_model_masks = []
                    
                    if step == 0:
                        for i, batch in enumerate(dataloader):
                            inp, index, mask = batch
                            indexes.append(index.cpu() if isinstance(index, torch.Tensor) else index)
                            imgs.append(inp[0].cpu())
                            masks.append(mask.cpu())

                            inp = [x.to(self.device) if isinstance(x, torch.Tensor) else x for x in inp]
                            mask = mask.to(self.device) 
                            logits = self.model(inp)
                            current_model_masks.append(logits.cpu())
                            
                            out = torch.sigmoid(logits) 
                            preds = (out >= thresh_tensor).float()
                            
                            for b in range(preds.shape[0]):
                                idx = index[b].item() if isinstance(index, torch.Tensor) else index[b]
                                summed_masks[idx] += preds[b]

                        
                            if order - 1 == step:
                                total_tp += (preds * mask).sum(dim=(0, 2, 3))
                                total_fp += (preds * (1 - mask)).sum(dim=(0, 2, 3))
                                total_fn += ((1 - preds) * mask).sum(dim=(0, 2, 3))

                            if i % 10 == 0:
                                print(f"step: {step} :: {i} batches processed")
                        prev_summed_masks = (summed_masks / self.norm_tensor).to(torch.float32)
                        model_masks = current_model_masks
            
                    else:
                        j = 1
                        for img, mod_mask, index, mask in zip(imgs, model_masks, indexes, masks):
                            # Move batch back to device from our CPU caches
                            img = img.to(self.device)
                            mod_mask = mod_mask.to(self.device)
                            mask = mask.to(self.device)
                            
                          
                            inp2 = prev_summed_masks[index].to(self.device)
                            
                            inp = (img, mod_mask, inp2)
                                                        
                            logits = self.model(inp)
                            current_model_masks.append(logits.cpu())
                            
                            out = torch.sigmoid(logits) 
                            preds = (out >= thresh_tensor).float()
                            
                            for b in range(preds.shape[0]):
                                idx = index[b].item() if isinstance(index, torch.Tensor) else index[b]
                                summed_masks[idx] += preds[b]

                            print(index)
                            
                            if order - 1 == step:
                                total_tp += (preds * mask).sum(dim=(0, 2, 3))
                                total_fp += (preds * (1 - mask)).sum(dim=(0, 2, 3))
                                total_fn += ((1 - preds) * mask).sum(dim=(0, 2, 3))

                            if j % 10 == 0:
                                print(f"step: {step} :: {j} batches processed")

                            j+=1
                        prev_summed_masks = (summed_masks / self.norm_tensor).to(torch.float32)
                        model_masks = current_model_masks

        denominator = 2 * total_tp + total_fp + total_fn
        
        f1_scores = torch.where(
            denominator != 0,
            (2 * total_tp) / denominator,
            torch.ones_like(denominator) 
        )
        
        return f1_scores.cpu().numpy()



In [14]:
thresholds = F1Metric(FinalModelCreationEntity())

Tested accelerator: cpu


In [15]:
thresholds.calculate_f1_direct(thresholds.val_dataloader, order = 0, thresholds =  np.load('data/stats/base_threshold.npy'))

0 batches processed
10 batches processed
20 batches processed
30 batches processed
40 batches processed
50 batches processed
60 batches processed
70 batches processed
80 batches processed
90 batches processed


array([0.7144809 , 0.63059014, 0.89355135], dtype=float32)

In [16]:
#[0.72257996, 0.64319706, 0.89688]
#new model[0.7215518 , 0.65346587, 0.8820928 ] thesh:  [0.928, 0.886, 0.9924]
#[0.7285136, 0.6523974, 0.8906582]

thresholds.calculate_f1_direct(thresholds.val_dataloader, order = 1, thresholds = np.load('data/stats/correction_threshold.npy'))

step: 0 :: 0 batches processed
step: 0 :: 10 batches processed
step: 0 :: 20 batches processed
step: 0 :: 30 batches processed
step: 0 :: 40 batches processed
step: 0 :: 50 batches processed
step: 0 :: 60 batches processed
step: 0 :: 70 batches processed
step: 0 :: 80 batches processed
step: 0 :: 90 batches processed


array([0.72257996, 0.64319706, 0.89688   ], dtype=float32)

In [17]:
thresholds.calculate_f1_direct(thresholds.val_dataloader, order = 2, thresholds =  np.load('data/stats/correction_threshold.npy'))

step: 0 :: 0 batches processed
step: 0 :: 10 batches processed
step: 0 :: 20 batches processed
step: 0 :: 30 batches processed
step: 0 :: 40 batches processed
step: 0 :: 50 batches processed
step: 0 :: 60 batches processed
step: 0 :: 70 batches processed
step: 0 :: 80 batches processed
step: 0 :: 90 batches processed
tensor([19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19])
tensor([19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19])
tensor([19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19])
tensor([19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19])
tensor([19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19])
tensor([19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19])
tensor([19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19])
tensor([19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19])
tensor([19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19])
tensor([118, 118, 118, 1

array([0.7195386 , 0.62936777, 0.8937925 ], dtype=float32)

In [18]:
thresholds.calculate_f1_direct(thresholds.val_dataloader, order = 3, thresholds =  np.load('data/stats/correction_threshold.npy'))

step: 0 :: 0 batches processed
step: 0 :: 10 batches processed
step: 0 :: 20 batches processed
step: 0 :: 30 batches processed
step: 0 :: 40 batches processed
step: 0 :: 50 batches processed
step: 0 :: 60 batches processed
step: 0 :: 70 batches processed
step: 0 :: 80 batches processed
step: 0 :: 90 batches processed
tensor([19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19])
tensor([19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19])
tensor([19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19])
tensor([19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19])
tensor([19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19])
tensor([19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19])
tensor([19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19])
tensor([19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19])
tensor([19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19])
tensor([118, 118, 118, 1

array([0.7058051 , 0.613838  , 0.88824445], dtype=float32)

In [2]:
df = pd.read_csv(CORRECTION_TEST_CSV)

In [3]:
len(df)

1584

In [4]:
len(df.collapsed_id.unique())

11

In [5]:
base_model_path = Path(f'data/models/base_model.ckpt')
correction_model_path=Path(f'data/models/correction_model.ckpt')
base_hparams = torch.load(base_model_path, 'cpu', weights_only= False)['hyper_parameters']
correction_hparams = torch.load(correction_model_path, 'cpu', weights_only= False)['hyper_parameters']

In [6]:
base_hparams

{'bce_loss_pos_weight': [66.76699651, 72.81312019, 130.64631503],
 'learning_rate': 0.0003885771767668683,
 'weight_decay': 0.006129038398536625,
 'inp_channels': 3,
 'first_conv_out_channels': 64,
 'num_classes': 3,
 'depth': 4,
 'n_encoder_conv_layers': 2,
 'n_decoder_conv_layers': 2,
 'kernel_sizes': [5, 3, 3, 5, 3, 5, 5, 3],
 'model_name': 'final_gpu-final'}

In [7]:
correction_hparams

{'bce_loss_pos_weight': [66.76699651, 72.81312019, 130.64631503],
 'learning_rate': 0.0005,
 'weight_decay': 0.001,
 'inp_channels': 7,
 'first_conv_out_channels': 32,
 'num_classes': 3,
 'depth': 3,
 'n_encoder_conv_layers': 2,
 'n_decoder_conv_layers': 2,
 'kernel_sizes': 3,
 'model_name': 'Error-correction-model-1'}

In [14]:
torch.load(base_model_path, 'cpu', weights_only= False)['epoch']

38

In [15]:
torch.load(correction_model_path, 'cpu', weights_only= False)['epoch']

6